# 03. Cloudera CML에서 vLLM 실행

이 노트북에서는 **Cloudera CML(Cloudera Machine Learning)** 환경의 구조를 이해하고,  
로컬(M2 Pro)과 CML 환경의 차이를 비교합니다.

---

## Cloudera CML 구조 이해

```
CML 프로젝트
│
├── Session (세션)
│   └── Jupyter Notebook, Terminal 등 대화형 작업
│   └── GPU/CPU 리소스 직접 선택 가능
│
├── Job (잡)
│   └── 배치 작업 스케줄링 (데이터 처리, 모델 학습 등)
│
└── Application (애플리케이션)
    └── 웹 서버를 외부에 공개 (우리 앱이 여기에 배포됨)
    └── 특정 포트(8100)로 외부 접속 가능
    └── cdsw-build.sh → 환경 구성
    └── cdsw-run.sh   → 서버 시작
```

---

## 로컬 vs CML 환경 비교

| 항목 | 로컬 (M2 Pro) | Cloudera CML |
|------|--------------|---------------|
| **장치** | CPU (Apple Silicon) | GPU (NVIDIA CUDA) |
| **모델 크기** | 3B (소형) | 7B 이상 (대형) |
| **vLLM dtype** | `float32` | `bfloat16` |
| **추론 속도** | 느림 (~5 tok/s) | 빠름 (~50+ tok/s) |
| **배포 방식** | Docker Compose | CML Application |
| **설정 파일** | `docker-compose.yml` | `cdsw-build.sh` / `cdsw-run.sh` |
| **접속 포트** | `localhost:8080` | CML이 제공하는 URL |

## Cell 1. cdsw-build.sh 역할 이해

CML에서 Application을 생성할 때 **자동으로 실행**되는 빌드 스크립트입니다.  
의존성 설치, 환경 구성, 프론트엔드 빌드 등을 수행합니다.

```bash
# cdsw-build.sh 내용 (우리 프로젝트)
#!/bin/bash
set -e  # 오류 발생 시 즉시 중단

echo "[1/2] Python 의존성 설치..."
pip install -r requirements/cloudera.txt

echo "[2/2] 빌드 완료!"
```

**CML에서 빌드가 실행되는 시점:**
1. Application 처음 생성할 때
2. "Rebuild" 버튼 클릭 시
3. 프로젝트 환경 재구성 시

## Cell 2. cdsw-run.sh 역할 이해

Application이 시작될 때 실행되는 스크립트입니다.  
vLLM 서버를 백그라운드로 기동하고, 그 다음 FastAPI 서버를 시작합니다.

```bash
# cdsw-run.sh 내용 (우리 프로젝트)
#!/bin/bash
set -e

echo "[1/2] vLLM 서버 시작 (백그라운드)..."
# & 기호로 백그라운드 실행
bash scripts/start_vllm.sh &

# vLLM 서버가 준비될 때까지 대기 (최대 120초)
echo "vLLM 서버 준비 대기 중..."
for i in $(seq 1 24); do
    sleep 5
    if curl -s http://localhost:8000/health > /dev/null 2>&1; then
        echo "vLLM 서버 준비 완료!"
        break
    fi
    echo "  대기 중... (${i}/24)"
done

echo "[2/2] FastAPI 앱 시작 (포트 8100)..."
# CML Application은 반드시 포트 8100을 사용해야 함
uvicorn app.main:app --host 0.0.0.0 --port 8100
```

**중요: CML Application 포트는 반드시 8100**  
CML이 외부 URL → 내부 포트 8100으로 트래픽을 라우팅합니다.

## Cell 3. GPU에서 vLLM 실행 — 명령어 비교

CPU와 GPU 실행 명령어를 나란히 비교합니다.  
변경되는 파라미터는 `--device`, `--dtype`, 그리고 사용 가능한 모델 크기입니다.

In [ ]:
# CPU와 GPU vLLM 실행 명령어를 비교 출력합니다
# (실제 실행이 아닌, 명령어 학습용 셀입니다)

cpu_command = """
# ===== 로컬 M2 Pro (CPU 모드) =====
python -m vllm.entrypoints.openai.api_server \\
  --model Qwen/Qwen2.5-3B-Instruct \\   # 소형 모델 (3B)
  --device cpu \\                         # CPU 실행
  --dtype float32 \\                      # CPU는 float32 사용
  --max-model-len 4096 \\                 # 메모리 절약을 위해 짧게 설정
  --port 8000

예상 속도: ~3~8 토큰/초 (매우 느림)
메모리 사용: ~8GB RAM
"""

gpu_command = """
# ===== Cloudera CML (GPU 모드) =====
python -m vllm.entrypoints.openai.api_server \\
  --model Qwen/Qwen2.5-7B-Instruct \\   # 대형 모델 (7B) 사용 가능
  --device cuda \\                        # GPU 실행
  --dtype bfloat16 \\                     # GPU는 bfloat16으로 속도+메모리 최적화
  --max-model-len 8192 \\                 # 더 긴 컨텍스트 처리 가능
  --tensor-parallel-size 1 \\             # GPU 1개 사용 (여러 개면 2,4 등으로 설정)
  --port 8000

예상 속도: ~50~100 토큰/초 (CPU 대비 10~20배)
메모리 사용: ~16GB GPU VRAM
"""

print(cpu_command)
print("-" * 60)
print(gpu_command)

print("\n💡 핵심 차이점 요약:")
print("  --device: cpu → cuda")
print("  --dtype: float32 → bfloat16  (메모리 절반, 속도 2배)")
print("  모델 크기: 3B → 7B  (GPU 메모리가 충분하면 더 큰 모델 가능)")

## Cell 4. CML GPU 리소스 할당 방법

CML에서 Session이나 Application에 GPU를 할당하는 방법을 설명합니다.

---

### Session에서 GPU 사용 (노트북 실험용)

```
CML 왼쪽 메뉴 → New Session
  ↓
Resource Profile 선택:
  - vCPU: 4개
  - Memory: 16GB
  - GPU: 1개  ← 여기서 GPU 선택
  ↓
Launch Session
```

### Application에서 GPU 사용 (서비스 배포용)

```
CML 왼쪽 메뉴 → Applications → New Application
  ↓
설정:
  - Name: beauty-fashion-ai
  - Script: cdsw-run.sh       ← 실행 스크립트
  - Resource Profile:
      vCPU: 4, Memory: 16GB, GPU: 1  ← GPU 할당
  ↓
Create Application
  ↓
자동으로 cdsw-build.sh 실행 후 cdsw-run.sh 시작
  ↓
외부 URL 자동 생성 (예: https://my-project.cml.example.com)
```

## Cell 5. 환경별 설정 파일 비교

같은 코드지만, 브랜치별로 다른 환경 설정 파일을 사용합니다.

In [ ]:
# 두 브랜치의 환경 설정 차이를 출력합니다

local_env = {
    "브랜치": "local-m2pro",
    "설정 파일": ".env.local",
    "VLLM_BASE_URL": "http://localhost:8000/v1",
    "MODEL_NAME": "Qwen/Qwen2.5-3B-Instruct",
    "DEVICE": "cpu",
    "DTYPE": "float32",
    "APP_PORT": "8080",
    "MAX_TOKENS": "300",
}

cloudera_env = {
    "브랜치": "cloudera-cml",
    "설정 파일": ".env.cloudera",
    "VLLM_BASE_URL": "http://localhost:8000/v1",   # 같은 값 (같은 서버 내부)
    "MODEL_NAME": "Qwen/Qwen2.5-7B-Instruct",      # 더 큰 모델
    "DEVICE": "cuda",                               # GPU
    "DTYPE": "bfloat16",                            # GPU 최적화
    "APP_PORT": "8100",                             # CML Application 포트
    "MAX_TOKENS": "500",                            # 더 긴 응답 허용
}

print(f"{'항목':<20} {'로컬 (M2 Pro)':<35} {'Cloudera CML':<35}")
print("-" * 90)

for key in local_env:
    local_val = local_env[key]
    cloud_val = cloudera_env[key]
    diff_marker = "  ← 다름" if local_val != cloud_val else ""
    print(f"{key:<20} {local_val:<35} {cloud_val:<35}{diff_marker}")

## 정리

**Cloudera CML에서 vLLM을 실행하는 흐름:**

```
1. CML에 프로젝트 생성 (cloudera-cml 브랜치 연결)
       ↓
2. Application 생성 + GPU 리소스 할당
       ↓
3. cdsw-build.sh 자동 실행 (pip install requirements/cloudera.txt)
       ↓
4. cdsw-run.sh 자동 실행
   ├── vLLM 서버 시작 (백그라운드, GPU 모드, 7B 모델)
   └── FastAPI 앱 시작 (포트 8100)
       ↓
5. CML이 외부 HTTPS URL 제공
       ↓
6. 브라우저에서 챗봇 사용
```

**로컬과의 핵심 차이:**
- `--device cpu` → `--device cuda` (GPU 사용)
- `float32` → `bfloat16` (메모리 효율화)
- 3B 모델 → 7B 모델 (더 높은 품질 응답)
- Docker Compose → CML Application (관리형 배포)

---

다음 단계: **`app/main.py`** 를 열어서 실제 FastAPI 코드를 확인하세요.